# Agent 7 — The Report Shows Its Work

Code merges duplicate claims and ranks by confidence; the model writes
prose **from the table and nothing else**; a final diff proves no number
crept in. Merging, ranking, and the diff run offline.

**No class API key?** The precomputed report is real-shaped; the diff —
the integrity check — runs on whatever report text you have.

In [ ]:
%pip install -q anthropic

In [ ]:
import os, getpass
# Ask your teacher for the class API key. It is never typed into a cell,
# never saved in the notebook - getpass keeps it out of your file.
try:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Class API key: ")
    HAVE_KEY = len(os.environ["ANTHROPIC_API_KEY"]) > 10
except Exception:
    HAVE_KEY = False
print("Key loaded." if HAVE_KEY else "No key - the notebook still teaches: precomputed outputs are shown below each live cell.")

In [ ]:
MODEL = "claude-opus-5"

def ask(prompt, system=None, max_tokens=1000):
    """One model call, plain text in and out."""
    import anthropic
    client = anthropic.Anthropic()
    kwargs = dict(model=MODEL, max_tokens=max_tokens,
                  messages=[{"role": "user", "content": prompt}])
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[-1].text

def get_json(prompt, tries=3):
    """Ask for JSON only; parse; re-ask on failure. The retry pattern from Build with LLMs."""
    import json as _json
    for attempt in range(tries):
        text = ask(prompt + "\n\nReply with ONLY valid JSON.")
        try:
            start = text.index("[") if "[" in text.split("{")[0] else text.index("{")
            return _json.loads(text[start:])
        except (ValueError, KeyError):
            continue
    raise RuntimeError("no valid JSON after retries")

## The verified claim table (lesson 6's output)

In [ ]:
VERIFIED = [
 {"claim": "The garden reached 60 plots in 2026", "verdict": "CONFIRMED", "metric": "plots-2026",
  "sources": ["cityparks.gov/report-2026"]},
 {"claim": "Now 60 plots after the spring expansion", "verdict": "CONFIRMED", "metric": "plots-2026",
  "sources": ["lakeview-news.com/garden-expands"]},
 {"claim": "48 plots as of 2023", "verdict": "CONFIRMED", "metric": "plots-2023",
  "sources": ["riverside-garden.org/history"]},
 {"claim": "48 member families in 2026", "verdict": "CONFIRMED", "metric": "families-2026",
  "sources": ["cityparks.gov/report-2026", "riverside-garden.org/about"]},
 {"claim": "31 member families in 2023", "verdict": "CONFIRMED", "metric": "families-2023",
  "sources": ["lakeview-news.com/roundup-2023"]},
 {"claim": "$15,000 expansion grant in 2025", "verdict": "CONFIRMED", "metric": "grant",
  "sources": ["cityparks.gov/grants-2025"]},
 {"claim": "22 families waitlisted", "verdict": "CONFIRMED", "metric": "waitlist",
  "sources": ["riverside-garden.org/join"]},
 {"claim": "Land lease runs through 2028", "verdict": "PLAUSIBLE", "metric": "lease",
  "sources": ["cityparks.gov/grants-2025"]},
]
QUARANTINE = [{"claim": "The garden has 600 plots", "reason": "3 sources state 60"}]

## Code merges and ranks

Same metric → one claim, sources pooled. Two independent sources agreeing
is evidence; the same fact counted twice is decoration.

In [ ]:
def merge(rows):
    merged = {}
    for r in rows:
        m = merged.setdefault(r["metric"], {"claim": r["claim"], "verdict": r["verdict"],
                                            "metric": r["metric"], "sources": []})
        m["sources"] = sorted(set(m["sources"]) | set(r["sources"]))
    out = list(merged.values())
    out.sort(key=lambda r: (r["verdict"] != "CONFIRMED", -len(r["sources"])))
    return out

table = merge(VERIFIED)
for r in table:
    print(f"{r['verdict']:10} {r['claim']}  ({len(r['sources'])} source{'s' if len(r['sources'])>1 else ''})")
assert len(table) == 7, "eight rows should merge to seven (two plots-2026 rows are one fact)"
assert len([r for r in table if r['metric'] == 'plots-2026'][0]['sources']) == 2

## The model writes — under orders

In [ ]:
WRITE_PROMPT = """Write a half-page research report answering:
'Is the Riverside Community Garden growing?'

Use ONLY the claims below. Cite sources in brackets after each factual
sentence. Hedge any claim marked PLAUSIBLE ('a single source suggests...').
End with a Limits section naming what the claims do not cover. Do not add,
round, or compute any number not present in the claims.

CLAIMS:
{claims}

QUARANTINED (mention in Limits): {quarantine}"""

PRECOMPUTED_REPORT = """Is the Riverside Community Garden growing?

Yes, on every measure a source covers. The garden reached 60 plots in 2026
[cityparks.gov/report-2026; lakeview-news.com/garden-expands], up from 48
in 2023 [riverside-garden.org/history]. Membership rose from 31 families in
2023 [lakeview-news.com/roundup-2023] to 48 in 2026 [cityparks.gov/report-2026;
riverside-garden.org/about]. A $15,000 expansion grant arrived in 2025
[cityparks.gov/grants-2025], and 22 families are waitlisted for plots
[riverside-garden.org/join]. A single source suggests the land lease runs
through 2028; treat that as unconfirmed [cityparks.gov/grants-2025].

Limits: no source states the garden's operating budget. The lease claim
rests on one page. One blog figure of 600 plots was refuted (three sources
state 60) and excluded."""

import json as _json
report = ask(WRITE_PROMPT.format(claims=_json.dumps(table, indent=1),
             quarantine=QUARANTINE)) if HAVE_KEY else PRECOMPUTED_REPORT
print(report)

## The number diff — nothing creeps in

Every number in the report must trace to a row. A number with no row is an
invention, no matter how reasonable it looks.

In [ ]:
import re
report_numbers = set(re.findall(r"\$?[\d,]+", report.replace("[", " [")))
report_numbers = {n for n in report_numbers if n.strip("$,").isdigit() and len(n.strip("$,")) < 6}
table_numbers = set()
for r in table:
    table_numbers |= set(re.findall(r"\$?[\d,]+", r["claim"]))
table_numbers |= {"600", "60"}   # quarantine numbers may appear in Limits

loose = {n for n in report_numbers if n not in table_numbers and n.strip("$,") not in
         {x.strip("$,") for x in table_numbers}}
print("numbers in report:", sorted(report_numbers))
print("unbacked numbers: ", sorted(loose) if loose else "none - every number has a row")
assert not loose, "a number with no claim row slipped into the report"

## Try it

1. Edit the precomputed report: change 22 to 25. Rerun the diff and watch
   it catch the edit.
2. Delete the Limits paragraph and read the report again. Same facts —
   what changed about how much you trust it?
3. **Build turn-in:** the number-to-row list for the full report, and the
   sentence you'd cut if you could keep only half.